# Step 0: Import requisite libraries

In [ ]:
import pandas as pd
import numpy as np
import xgboost as xgb
from sklearn import preprocessing, ensemble, tree
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split, TunedThresholdClassifierCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.pipeline import Pipeline
import matplotlib as plt
import seaborn

# Step 1: Import and cleaning of data

In [ ]:
df = pd.read_csv("Baza_customer_Telecom_v2.csv")


,PID,CRM_PID_Value_Segment,EffectiveSegment,Billing_ZIP,KA_name,Active_subscribers,Not_Active_subscribers,Suspended_subscribers,Total_SUBs,AvgMobileRevenue,AvgFIXRevenue,TotalRevenue,ARPU,CHURN
0,123759242,Bronze,SOHO,6000.0,VM,2,NaN,NaN,2,40.17,0.0,40.17,NaN,No
1,126145737,Bronze,SOHO,6400.0,VM,3,NaN,NaN,3,40.17,0.0,40.17,13.39,No
2,123506355,Bronze,SOHO,6000.0,DI,2,3.0,NaN,5,40.17,0.0,40.17,20.09,No
3,112595585,Bronze,SOHO,4400.0,MT,1,2.0,NaN,3,40.17,0.0,40.17,40.17,No
4,115097935,Iron,SOHO,4000.0,AD,2,1.0,NaN,3,40.17,0.0,40.17,20.09,No
5,108684074,Gold,VSE,6600.0,RJ,2,11.0,NaN,13,40.17,0.0,40.17,20.09,No
6,112504678,Bronze,SOHO,4644.0,MT,4,NaN,NaN,4,40.33,0.0,40.33,10.08,Yes
7,833121152,Iron,SOHO,6100.0,VM,5,2.0,NaN,7,40.33,0.0,40.33,8.07,No
8,123654309,Iron,SOHO,6100.0,DI,3,NaN,NaN,3,40.33,0.0,40.33,13.44,No
9,123688980,Bronze,SOHO,6280.0,AD?,1,NaN,NaN,1,40.33,0.0,40.33,40.33,No


In [729]:
print(f"Length of dataset before cleaning: {len(df)}")
# Cleaning for PID
df = df.loc[df["PID"].notnull() & df["PID"].str.fullmatch(r"[0-9]*(SMB)?")] # Some PIDs were scientific notation (invalid); valid format is integer optionally followed by "SMB"

# Cleaning for value segment
df = df.loc[df["CRM_PID_Value_Segment"].notnull()]
df["CRM_PID_Value_Segment"] = df["CRM_PID_Value_Segment"].replace("Sliver", "Silver") # Fix assumed typo in one entry

# Cleaning for effective segment
df = df.loc[df["EffectiveSegment"].notnull()]

# Cleaning for ZIP
df = df.loc[df["Billing_ZIP"] % 1 == 0] # Remove non-integer ZIP codes (invalid; includes nulls)

# Cleaning for KA_name
df = df.loc[df["KA_name"].notnull()]
df["KA_name"] = df["KA_name"].replace("AD?", "AD") # Grouped AD? with AD under assumption that unsure attribution is correct

# Cleaning for active_subscribers
df = df.loc[df["Active_subscribers"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for not_active_subscribers
df["Not_Active_subscribers"] = df["Not_Active_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none inactive; minimum non-NaN value is 1)

# Cleaning for suspended_subscribers
df["Suspended_subscribers"] = df["Suspended_subscribers"].fillna(0) # Fill nulls with 0 (assuming nulls indicate none suspended; minimum non-NaN value is 1)

# Cleaning for total_SUBs
df = df.loc[df["Total_SUBs"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for AvgMobileRevenue
df = df.loc[df["AvgMobileRevenue "].notnull()] # Redundant (no nulls) but included for consistency
df = df.rename(columns={"AvgMobileRevenue ": "AvgMobileRevenue"}) # Remove trailing space from column name

# Cleaning for AvgFIXRevenue
df = df.loc[df["AvgFIXRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for TotalRevenue
df = df.loc[df["TotalRevenue"].notnull()] # Redundant (no nulls) but included for consistency

# Cleaning for ARPU
df = df.loc[df["ARPU"].notnull()]

# Cleaning for CHURN
df = df.loc[df["CHURN"].isin(["No", "Yes"])] # Remove invalid CHURN values (only valid values are "No" and "Yes"); redundant (no invalids) but included for consistency

print(f"Length of dataset after cleaning: {len(df)}") # Small length change, hence valid to discard nulls/invalid entries

# Source - https://stackoverflow.com/a/48035423
# Posted by BENY, modified by community. See post 'Timeline' for change history
# Retrieved 2026-09-02, License - CC BY-SA 4.0

df.apply(lambda x : pd.factorize(x)[0]).corr(method='pearson', min_periods=1)
df["CHURN"].value_counts()

Length of dataset before cleaning: 8453
Length of dataset after cleaning: 8419


CHURN
No     7876
Yes     543
Name: count, dtype: int64

# Step 2: Data Preprocessing

In [730]:
X = df.drop(columns=["CHURN"])
y = df["CHURN"].map({"No": 0, "Yes": 1}) # Convert CHURN to binary values (0 for No, 1 for Yes)

X["PID_is_SMB"] = X["PID"].str.endswith("SMB").astype("string") # Create binary feature indicating if PID ends with "SMB"
X["PID"]=X["PID"].str.replace("SMB", "").astype(int) # Remove "SMB" from PID and convert to integer

# Definition of categorical and numerical features for preprocessing
cat_features = ["CRM_PID_Value_Segment", "EffectiveSegment", "KA_name", "PID_is_SMB"]
num_features = ["Billing_ZIP", "Active_subscribers", "Not_Active_subscribers", "Suspended_subscribers", "Total_SUBs", "AvgMobileRevenue", "AvgFIXRevenue", "TotalRevenue", "ARPU"]

# Selection of features for model training

# train_features = cat_features + num_features

train_features = [
    "PID_is_SMB",
    "CRM_PID_Value_Segment", 
    "Active_subscribers",
    "AvgMobileRevenue",
    "TotalRevenue"
    ]

X = X[train_features]

# Definition of ColumnTransformer for preprocessing
c_transformer = ColumnTransformer(
    transformers=[
        ("categorical", preprocessing.OneHotEncoder(handle_unknown="ignore"), list(set(cat_features) & set(train_features)) ),
        ("numerical", "passthrough", list(set(num_features) & set(train_features)) )
    ]
)

print(f"Column transformer has categorical features {list(set(cat_features) & set(train_features))} and numerical features {list(set(num_features) & set(train_features))}")

# Model pipeline
clf_p = Pipeline([
    ("preprocessor", c_transformer),
    # ("classifier", ensemble.RandomForestClassifier(
    #     n_estimators=500,
    #     class_weight="balanced",
    #     n_jobs=-1,
    #     random_state=42))
    ("classifier", tree.DecisionTreeClassifier(
        class_weight="balanced",
        splitter="best",
        max_depth=None,
        random_state=42))
])

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Fit model
clf_p.fit(X_train, y_train)

# Predict on test set
y_pred_p = clf_p.predict_proba(X_test)
t_df = pd.DataFrame(X_test)
t_df["Predicted_Churn_Probability"] = y_pred_p[:, 1] # Get probabilities for positive class (CHURN = Yes)
t_df["Predicted_Churn"] = (t_df["Predicted_Churn_Probability"] >= 0.5).astype(int) # Convert probabilities to binary predictions using threshold of 0.5
t_df["Actual_Churn"] = y_test.values # Add actual CHURN values for comparison
t_df["Prediction_type"] = np.select(
    [
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 1),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 1) & (t_df["Actual_Churn"] == 0),
        (t_df["Predicted_Churn"] == 0) & (t_df["Actual_Churn"] == 1)
    ],
    [
        "True Positive",
        "True Negative",
        "False Positive",
        "False Negative"
    ],
    default="Unknown"
)

# Evaluate model performance
y_pred = y_pred_p[:, 1] >= 0.4
print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
t_df.to_csv("Predicted_Churn_Results.csv", index=False) # Save predictions to CSV for further analysis
t_df[t_df["Prediction_type"] == "True Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis


Column transformer has categorical features ['CRM_PID_Value_Segment', 'PID_is_SMB'] and numerical features ['Active_subscribers', 'TotalRevenue', 'AvgMobileRevenue']
              precision    recall  f1-score   support

           0       0.94      0.92      0.93      1575
           1       0.12      0.16      0.14       109

    accuracy                           0.87      1684
   macro avg       0.53      0.54      0.53      1684
weighted avg       0.89      0.87      0.88      1684

[[1453  122]
 [  92   17]]


,PID_is_SMB,CRM_PID_Value_Segment,Active_subscribers,AvgMobileRevenue,TotalRevenue,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
5636,False,Bronze,10,74.00,74.00,1.000000,1,1,True Positive
4446,False,Gold,28,284.33,284.33,1.000000,1,1,True Positive
4155,False,Gold,13,244.17,244.17,1.000000,1,1,True Positive
3716,False,Platinum,11,203.67,203.67,1.000000,1,1,True Positive
4165,False,Gold,11,245.50,245.50,1.000000,1,1,True Positive
3436,False,Platinum,1,181.17,181.17,1.000000,1,1,True Positive
6803,False,Silver,5,178.50,178.50,1.000000,1,1,True Positive
7175,False,Bronze,2,63.17,63.17,1.000000,1,1,True Positive
2449,False,Bronze,13,126.17,126.17,1.000000,1,1,True Positive
2981,False,Silver,14,153.67,153.67,1.000000,1,1,True Positive


In [731]:
t_df[t_df["Prediction_type"] == "False Positive"].sort_values(by="Predicted_Churn_Probability", ascending=False).head(20) # Sort by predicted probability for easier analysis

,PID_is_SMB,CRM_PID_Value_Segment,Active_subscribers,AvgMobileRevenue,TotalRevenue,Predicted_Churn_Probability,Predicted_Churn,Actual_Churn,Prediction_type
4105,False,Gold,8,238.50,238.50,1.0,1,0,False Positive
5780,False,Bronze,1,80.33,80.33,1.0,1,0,False Positive
4351,False,Gold,16,270.00,270.00,1.0,1,0,False Positive
5074,False,Platinum,8,441.75,441.75,1.0,1,0,False Positive
1273,False,Bronze,5,83.00,83.00,1.0,1,0,False Positive
693,False,Bronze,5,64.50,64.50,1.0,1,0,False Positive
2270,False,Bronze,8,118.33,118.33,1.0,1,0,False Positive
5161,False,Platinum,8,496.33,496.33,1.0,1,0,False Positive
3566,False,Silver,6,190.00,190.00,1.0,1,0,False Positive
3572,False,Gold,22,190.17,190.17,1.0,1,0,False Positive


In [732]:
# Testing per-group
test_segment = "Bronze"
segment_test = y_test[X_test["CRM_PID_Value_Segment"] == test_segment]
segment_pred = y_pred[X_test["CRM_PID_Value_Segment"] == test_segment]

print(classification_report(segment_test, segment_pred))

              precision    recall  f1-score   support

           0       0.94      0.93      0.94       703
           1       0.15      0.16      0.15        49

    accuracy                           0.88       752
   macro avg       0.54      0.55      0.55       752
weighted avg       0.89      0.88      0.89       752

